# Phase 4 Step 2 — Popularity baseline
Validation-only, point-in-time, full-catalog evaluation. **Lockbox recommendation performance is not inspected.**

## Objective and frozen recommendation contract
Rank the next-interaction item for each eligible visitor. History and item availability are limited to events at or before `T`; the single target is strictly after `T`. Visitors require two prior interactions, seen items remain eligible, and cold targets remain misses.

In [ ]:
from pathlib import Path
import json, pandas as pd
from marketmind.recommendations.baseline import run_validation
ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
ARTIFACTS = ROOT / 'reports/recommendation/artifacts'


## Validation instance construction
`next_item_instances` creates exactly 8,083 visitors at `2015-08-17 03:00 UTC`; target timestamps are before `2015-09-01 03:00 UTC`.

In [ ]:
# Reproducible execution; the runner removes lockbox rows before evaluation.
summary = run_validation(ROOT/'data/raw/retailrocket/events.csv', ARTIFACTS)
instances = pd.read_csv(ARTIFACTS/'validation_instances.csv')
assert len(instances) == 8083 and instances.visitorid.is_unique


## Point-in-time catalog and popularity baseline
At each distinct `T`, rank items by cumulative interaction count through `T`, descending, then numeric item ID ascending. There are no event or recency weights and no visitor-specific filtering.

In [ ]:
top20 = pd.read_csv(ARTIFACTS/'popularity_top20.csv')
top20.head()


## Full-catalog ranking and metrics
Only top 20 IDs are stored; no dense 8,083 × 211,905 matrix exists. Primary metric is macro single-target NDCG@10. Recall and HitRate are identical here; Precision@K is `hit/K`.

In [ ]:
pd.Series(summary['overall_metrics'], name='macro_validation_metric')


## Target-event, repeat/novel, cold/warm, and history-depth slices

In [ ]:
{name: pd.DataFrame(values).T[['instances','ndcg_at_10','hit_rate_at_10','hit_rate_at_20']] for name, values in summary['slices'].items()}


## Catalog coverage, concentration, and computational profile

In [ ]:
display(pd.DataFrame(summary['coverage']).T)
display(pd.Series(summary['concentration']))
display(pd.Series(summary['compute']))


## Sanity checks and limitations
The runner asserts instance count, strict target time, validation end, unique targets, deterministic top-K, time-safe availability, and removal of lockbox rows. Popularity is non-personalized, view-dominated, low-coverage, and cannot retrieve cold items.

## Decision
The canonical pipeline and frozen baseline are established. Step 3 may test a simple personalized collaborative baseline without changing this contract. Lockbox rankings remain untouched.